# Lab 8: Testing differences in reward anticipation

**Synthetic dataset v1.0.1**

Does the average reward-anticipation contrast differ from zero? Does it differ between reward sizes within people, or between separate substance-use reporting groups? Use three t-test designs to answer these questions with estimates and confidence intervals.

**Learning objectives.** By the end of this lab, you should be able to:

- identify the outcome, its units (PSC), the analysis unit, and the denominator for each comparison;
- choose between one-sample, paired, and independent (Welch) t procedures, and explain which variability enters each standard error;
- calculate and interpret a t-based 95% confidence interval and two-sided test, including what a p-value does and does not mean;
- read each estimate and interval against a stated example of a difference worth noticing, keeping statistical evidence separate from size;
- explain how quality exclusions, what a task-fMRI contrast measures, and observational grouping limit the claims you can make.

**How this lab connects.** Exercise 8 described invented people's left nucleus accumbens contrasts. Today's warm-up practiced reading an estimate against its uncertainty. This lab asks population questions about 3,994 synthetic participants with the same kinds of contrasts.

Complete **three Your Turns**, one per test. Supplied code loads and checks the data and produces plots. Interpretation prompts prepare you for the **Lab 8 Canvas assessment**; notebook notes are optional and no notebook upload is required.

The records are synthetic, not real ABCD participant data. No personal substance-use disclosure is requested. Reading: Vu and Harrington §§5.1–5.3.

Use Copilot for hints or debugging after your attempt. Complete Canvas independently without generative AI answering, drafting, rewriting, or checking responses. Approved accessibility, spelling/grammar, and instructor-authorized translation tools remain permitted.


## Section 1. Set up the analysis

Set **CHOICE** to cannabis or alcohol before examining associations and retain that choice. The first two analyses are common to everyone; only the independent-group comparison depends on the substance.

All three tests are **two-sided**, with **alpha=.05** and **95% confidence intervals**. Proceed with every planned comparison regardless of earlier p-values. Do not change direction, exclusions, or substance choice to obtain a preferred result.

**A difference worth noticing.** Read the size of each estimate alongside its p-value. For all three comparisons, this lab uses **0.02 PSC** in either direction as a hypothetical difference worth noticing. It is a teaching value fixed before any results, like Lab 7's 0.5-day criterion. It is not an established scientific or clinical threshold. Section 5 asks what each interval says about differences of that size; Lab 9 returns to how such a value is justified.

Run the supplied setup. It loads one synthetic wave, checks the file and unique participant IDs, and records the questions. Each analysis below introduces its measure and eligibility rules where they are used.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
try:
    from IPython.display import display, Markdown
except ImportError:  # Plain-Python review; Jupyter uses its normal rich display.
    def display(*objects):
        for obj in objects:
            print(obj)
    Markdown = str

CHOICE = 'cannabis'  # Set 'cannabis' or 'alcohol' BEFORE viewing results.
assert CHOICE in ('cannabis', 'alcohol')
ALPHA = 0.05
EXAMPLE_DIFFERENCE = 0.02  # PSC; hypothetical teaching value fixed before results
NOTEBOOK_VERSION = '3.4'
DATASET_VERSION = '1.0.1'
# Fingerprint of the v1.0.1 CSV with line endings normalized, so the check passes
# whether the file was stored with Windows or Unix line endings.
EXPECTED_SHA256 = 'ebb65a6b98e834f97d006ec175a855905f8c3bfc0f74befff240fd08a8b96899'
# Optional override: a folder containing M8_M9_STUDENT.csv and its documentation.
DATA_DIR_OVERRIDE = None
if DATA_DIR_OVERRIDE is not None:
    candidates = [Path(DATA_DIR_OVERRIDE)]
else:
    candidates = [base / 'data' / 'labs_08_09' for base in (Path.cwd(), *Path.cwd().parents)]
DATA = next((p for p in candidates if (p / 'M8_M9_STUDENT.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Cannot find data/labs_08_09/M8_M9_STUDENT.csv. Open this notebook from the '
                            'labs/ folder of the course repository, or set DATA_DIR_OVERRIDE.')
csv_path = DATA / 'M8_M9_STUDENT.csv'
csv_sha256 = hashlib.sha256(csv_path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()
assert csv_sha256 == EXPECTED_SHA256, ('This CSV differs from the v1.0.1 release. Restore the original '
                                       'file from the repository; do not edit it.')
df = pd.read_csv(csv_path, dtype={'participant_id':str, 'instructional_wave_id':str})
assert len(df) == 5000 and df.participant_id.notna().all() and df.participant_id.is_unique
assert df.instructional_wave_id.nunique() == 1
assert df.instructional_tlfb_window_days.eq(365).all()
POOLED = 'instructional_mid_nac_arvn_bilateral'
LARGE = 'instructional_mid_nac_alrvn_bilateral'
SMALL = 'instructional_mid_nac_asrvn_bilateral'
MID = 'instructional_mid_analysis_eligible'
ELIGIBLE = f'instructional_m8_{CHOICE}_analysis_eligible'
RESOLVED = f'instructional_{CHOICE}_report_resolved'
GROUP = f'instructional_{CHOICE}_any_detailed'
OTHER = 'alcohol' if CHOICE == 'cannabis' else 'cannabis'
OTHER_GROUP = f'instructional_{OTHER}_any_detailed'
KEYS = ['participant_id', 'instructional_wave_id']
pd.set_option('display.precision', 5)
analysis_plan = pd.DataFrame([
    ['One-sample', 'Mean pooled reward contrast vs zero', 'mean - 0'],
    ['Paired', 'Reward size within people', 'large - small'],
    ['Independent', f'Pooled contrast between {CHOICE} groups', 'any - zero'],
], columns=['design','question','direction'])
analysis_plan['example_difference'] = f'±{EXAMPLE_DIFFERENCE} PSC (hypothetical)'
display(analysis_plan)
print('Choice:', CHOICE, '| Dataset:', DATASET_VERSION)
print('Documentation folder:', DATA)
for filename in ['README.md','M8_M9_DATA_DICTIONARY.md','M8_M9_ANALYSIS_GUIDE.md']:
    assert (DATA / filename).is_file(), f'Missing student documentation: {filename}'
    relative = Path(os.path.relpath(DATA / filename, Path.cwd())).as_posix().replace(' ', '%20')
    display(Markdown(f'[{filename}]({relative})'))


## Section 2. Does the mean reward contrast differ from zero?

### Define the outcome and population question

The monetary incentive delay (MID) task measures responses during anticipation of possible rewards. Our outcome is a modeled nucleus accumbens **reward-minus-neutral contrast in percent signal change (PSC)**. It averages large- and small-reward contrasts and the two hemispheres equally. This **pooled bilateral** measure is a course derivative, not raw BOLD or a direct dopamine measurement.

Each eligible person contributes one pooled value. For example, bilateral large-minus-neutral = +0.30 PSC and small-minus-neutral = +0.10 PSC give a pooled value of (+0.30 + +0.10)/2 = +0.20 PSC. The contrast is a difference on the modeled percent-signal scale, not a percentage relative to the neutral estimate.

**H₀: μ = 0; Hₐ: μ ≠ 0.** Here μ is the population mean pooled contrast under the teaching model. Zero means no average condition contrast, not absence of brain signal. A nonzero sample mean alone does not settle the population question.

Use all MID-eligible records. No substance report is required for this question or the paired question. The supplied audit distinguishes unavailable imaging from failed MID quality; missing reports are not filled with zero. These synthetic records illustrate model-based inference, not a probability sample of all adolescents. We are not drawing a smaller sample from the file, and no finite-population correction is used.


### See the region behind each number

Each contrast value summarizes one brain region, the **nucleus accumbens (NAc)**. The `aseg` in the field names refers to an automated subcortical segmentation, which outlines the region in each person's structural scan. The value is the modeled response averaged across the voxels inside that outline: one number per person, per contrast, per hemisphere.

This release keeps the official field names, so you can look up a field from this notebook's checks. Here is the left-hemisphere large-reward field, `mr_y_tfmri__mid__alrvn__aseg__ab__lh_beta`, in the DEAP data dictionary:

![DEAP data-dictionary entry for mr_y_tfmri__mid__alrvn__aseg__ab__lh_beta: average beta weight for the MID task, anticipated large reward versus neutral, all runs, in the left accumbens area. Unit: percent signal change (PSC). Table: mr_y_tfmri__mid__alrvn__aseg.](../assets/l8_deap_mid_alrvn_lh_field.png)

![DEAP brain image for the same field: a translucent 3-D brain with the left nucleus accumbens highlighted in blue.](../assets/l8_deap_mid_alrvn_lh_roi.png)

*Source: DEAP data dictionary screenshots supplied by the instructor, November 2026. The records in this lab are synthetic values that reuse this field's name and definition; they are not ABCD participant data.*

The highlighted shape shows **where** the voxels are, not the signal itself. Exercise 8 used this left-hemisphere value. This lab averages it with the matching right-hemisphere field (`mr_y_tfmri__mid__alrvn__aseg__ab__rh_beta`) and then averages the large- and small-reward contrasts.

**Optional:** in the [DEAP data dictionary](https://abcd.deapscience.com/#/my-datasets/create-dataset), search for the field name and open its brain image. The image is interactive: rotate it to see where the region sits. DEAP may ask you to sign in or may be unavailable; the screenshots above are all you need for this lab.


In [ ]:
mid_data = df.loc[df[MID].eq(1)].copy()
assert mid_data[[POOLED,LARGE,SMALL]].notna().all().all()
eligibility_audit = pd.DataFrame([
    ['Starting records', len(df)],
    ['Imaging unavailable', int(df.instructional_mid_eligibility_reason.eq('imaging_unavailable').sum())],
    ['Failed MID primary quality', int(df.instructional_mid_eligibility_reason.eq('failed_primary_quality').sum())],
    ['MID eligible: one-sample and paired', len(mid_data)],
], columns=['stage','n'])
assert eligibility_audit.loc[1:3,'n'].sum() == len(df)
# Kept separately so the Section 4 cell can be re-run without duplicating rows.
common_eligibility_audit = eligibility_audit.copy()
display(eligibility_audit)
# Check the instructional averaging rule from the hemisphere analogs.
for word, field in [('alrvn', LARGE), ('asrvn', SMALL)]:
    left = f'mr_y_tfmri__mid__{word}__aseg__ab__lh_beta'
    right = f'mr_y_tfmri__mid__{word}__aseg__ab__rh_beta'
    assert np.allclose(mid_data[field], (mid_data[left] + mid_data[right]) / 2, atol=1e-8)
assert np.allclose(mid_data[POOLED], (mid_data[LARGE] + mid_data[SMALL]) / 2, atol=1e-8)


### Exclusions are research decisions

The audit removes two kinds of records before the first two analyses: people with no usable MID imaging, and people whose MID data failed the primary quality rule. Quality rules commonly reflect problems such as head motion. The release documentation fixes these rules before any results. Responsible reporting:

- states how many records were excluded at each step and why;
- notes that excluded people may differ from those analyzed, because scan quality can be related to characteristics such as age or attention, so the analyzed group may not represent everyone who was scanned;
- never adds back, drops, or fills in people to obtain a preferred result. A missing contrast is not replaced with a mean or with zero.

Imaging also asks a lot of participants, who must lie still in the scanner for an extended session, and raw brain images can contain identifying features such as the face. That is one reason shared files like this one hold derived regional values rather than raw images or lists of excluded participants. These records are synthetic; the same practices apply to real data. Canvas Question 5 uses this explanation and `eligibility_audit`.


### Demo: compare the difference with its uncertainty

**SE = sample SD / √n** estimates uncertainty in the mean; SD describes variation among people. The test statistic is **t = (mean − 0) / SE**, a distance from zero in standard-error units. A 95% interval is **mean ± t* × SE**, using **df=n−1**.

When population SD is unknown, the t reference accounts for estimating it. Its tails are heavier than the normal reference, especially at small n. Predict which interval will be wider for the supplied invented example: n=10, mean=0.12 PSC, SD=0.20 PSC. Then compare it with the worked calculation for the actual eligible file.


In [ ]:
demo_n, demo_mean, demo_sd = 10, .12, .20
demo_se = demo_sd / np.sqrt(demo_n)
tstar = stats.t.ppf(.975, demo_n-1)
zstar = stats.norm.ppf(.975)
small_sample_interval_demo = pd.DataFrame([
    ['t (unknown population SD)', tstar, demo_mean-tstar*demo_se, demo_mean+tstar*demo_se],
    ['z (teaching comparison)', zstar, demo_mean-zstar*demo_se, demo_mean+zstar*demo_se],
], columns=['reference','cutoff','low_PSC','high_PSC'])
display(small_sample_interval_demo)
print('t margin / z margin:', round(tstar/zstar, 3))
# Worked calculation for the common eligible sample.
x = mid_data[POOLED]
n = len(x); mean = x.mean(); sd = x.std(ddof=1); se = sd/np.sqrt(n); dfree = n-1
manual_t = mean/se
manual_low = mean - stats.t.ppf(.975, dfree)*se
manual_high = mean + stats.t.ppf(.975, dfree)*se
display(pd.DataFrame([dict(n=n,mean_PSC=mean,SD=sd,SE=se,df=dfree,t=manual_t,
                           low_PSC=manual_low,high_PSC=manual_high)]))


### Your Turn 1: calculate the interval and test

Complete two calls: `stats.t.interval` with confidence=.95, df=dfree, loc=mean, and scale=se; and `stats.ttest_1samp` with x and popmean=0. The default alternative is two-sided.

The **p-value** is the probability, under H₀ and its model assumptions, of a statistic at least as extreme in either direction. It is not the probability H₀ is true. A 95% interval procedure covers the population mean in 95% of repeated samples under its assumptions; the interval does not contain 95% of individuals.

Optional Copilot prompt after trying: “Explain the role of loc and scale in a t interval using a different invented example. Give one hint at a time, not my code.”


In [ ]:
# Your Turn 1
one_ci = None  # stats.t.interval(...)
one_test = None  # stats.ttest_1samp(...)


In [ ]:
assert one_ci is not None and one_test is not None, 'Complete both calls.'
assert np.allclose(one_ci, [manual_low,manual_high]), ('The interval does not match the worked calculation. '
    'Check confidence=.95, df=dfree, loc=mean, and scale=se (the SE, not the SD).')
assert np.isclose(one_test.statistic, manual_t), 'The t statistic does not match. Use x with popmean=0.'
assert np.isclose(one_test.pvalue, 2*stats.t.sf(abs(manual_t), dfree), rtol=1e-8, atol=0.0), 'Keep the default two-sided alternative.'
one_sample_result = pd.DataFrame([dict(n=n,mean_PSC=mean,SD=sd,SE=se,df=dfree,
    estimate_PSC=mean,low_PSC=one_ci[0],high_PSC=one_ci[1],t=one_test.statistic,p=one_test.pvalue)])
display(one_sample_result)
print('Large-sample t cutoff:', round(stats.t.ppf(.975,dfree),5), '| z cutoff:', round(zstar,5))
fig, ax = plt.subplots(figsize=(6,2.8))
ax.errorbar(mean,0,xerr=[[mean-one_ci[0]],[one_ci[1]-mean]],fmt='o',color='#13678A',capsize=6)
ax.axvline(0,color='black',linestyle='--'); ax.set(yticks=[0],yticklabels=['Pooled reward − neutral'],xlabel='Mean contrast (PSC)',title='Synthetic cohort: estimate and 95% CI')
one_sample_plot = fig
fig.tight_layout(); plt.show()


### Interpret and prepare for Canvas

Read the mean and interval in PSC. Does the interval include zero, and does the two-sided p-value give the corresponding decision at .05? Why are the t and z cutoffs close for this n?

A large sample does not remove selection or dependence problems. Interpret the result under the stated model, not as a claim about all adolescents.

**Have ready:** the mean, interval, and what the p-value is not. Questions 1, 4, and 10 are selected response. Notebook notes are optional.

For the Canvas assessment, Question 1 uses the outcome definition and hypotheses at the start of this section; Questions 4 and 10 (p-value interpretation) use `one_sample_result`. The margin of error is the distance from the estimate to either interval endpoint.


## Section 3. Does reward size change the contrast within people?

Each row already contains one person's **bilateral large-minus-neutral** and **bilateral small-minus-neutral** contrasts. Subtract small from large within that row. The common neutral term cancels within the same person, model, and scale. No new join is needed.

**H₀: μdifference = 0; Hₐ: μdifference ≠ 0.** The statistic asks whether the population mean within-person difference is zero. Even if a scientific prediction favors a positive difference, keep the prespecified two-sided test.

### Demo: one difference per person

If one person's large contrast is +0.30 and small is +0.10 PSC, their difference is +0.20 PSC. Another person's difference can be negative. Average these differences to estimate the population mean difference.

The paired test is one-sample inference on those differences: **SE = SD of differences / √number of pairs**. It preserves the relationship between the two measurements from each person. Combining two independent-group variances would ignore that relationship. n counts people with complete pairs, not twice as many observations.

The supplied cell selects the two same-row columns and checks completeness and identity.


In [ ]:
paired_data = mid_data[KEYS + [LARGE, SMALL]].sort_values(KEYS).reset_index(drop=True)
assert paired_data.participant_id.is_unique
assert paired_data.instructional_wave_id.nunique() == 1
assert paired_data[[LARGE, SMALL]].notna().all().all()
assert len(paired_data) == len(mid_data)
print('Complete paired participants:', len(paired_data))


### Your Turn 2: calculate differences and run the paired test

Subtract the small column from the large column in `paired_data`. Then call `stats.ttest_rel` with the large column first and small second. The supplied check compares your result with a one-sample test of the differences.

Before running, identify which SD belongs in the paired SE: the SD of large values, the SD of small values, or the SD of within-person differences.


In [ ]:
# Your Turn 2
paired_differences = None  # one large-minus-small difference per person
paired_test = None  # stats.ttest_rel(...)


In [ ]:
assert paired_differences is not None and paired_test is not None, 'Complete both expressions.'
reference_diff = mid_data.sort_values(KEYS)['instructional_mid_nac_large_minus_small'].to_numpy()
assert np.allclose(paired_differences, reference_diff, atol=1e-8), ('The differences do not match. '
    'Use the two columns of paired_data and subtract small from large.')
diff_test = stats.ttest_1samp(paired_differences, popmean=0)
assert np.isclose(paired_test.statistic, diff_test.statistic), ('The paired test does not match. '
    'Pass the large column first and the small column second to stats.ttest_rel.')
assert np.isclose(paired_test.pvalue, diff_test.pvalue, rtol=1e-8, atol=0.0), 'Keep the default two-sided alternative.'
pair_n = len(paired_differences); pair_mean = paired_differences.mean()
pair_sd = paired_differences.std(ddof=1); pair_se = pair_sd/np.sqrt(pair_n)
pair_ci = stats.t.interval(.95, pair_n-1, loc=pair_mean, scale=pair_se)
paired_result = pd.DataFrame([dict(n=pair_n,estimate_PSC=pair_mean,SD=pair_sd,SE=pair_se,df=pair_n-1,
    low_PSC=pair_ci[0],high_PSC=pair_ci[1],t=paired_test.statistic,p=paired_test.pvalue)])
paired_diagnostic = paired_differences.describe(percentiles=[.01,.25,.5,.75,.99]).to_frame('difference_PSC')
display(paired_diagnostic); display(paired_result)
fig, axes = plt.subplots(1,2,figsize=(9,3.5))
axes[0].hist(paired_differences,bins=35,color='#13678A',edgecolor='white')
axes[0].axvline(0,color='black',linestyle='--'); axes[0].set(xlabel='Within-person large − small (PSC)',ylabel='People',title='Check the differences, not separate raw columns')
axes[1].errorbar(pair_mean,0,xerr=[[pair_mean-pair_ci[0]],[pair_ci[1]-pair_mean]],fmt='s',capsize=6,color='#9A4D14')
axes[1].axvline(0,color='black',linestyle='--'); axes[1].set(yticks=[],xlabel='Mean paired difference (PSC)',title='Estimate and 95% CI')
paired_plot = fig
fig.tight_layout(); plt.show()


### Interpret and prepare for Canvas

Inspect the distribution of **differences**, including its tails and influential observations. Do not remove extremes to improve significance or use a normality-test p-value as an automatic method selector. The t procedure is exact for independent normal differences and often approximate otherwise.

Explain the mean difference and its interval. Does the average ordering apply to everyone? What does the p-value tell you under a zero-mean-difference null?

**Have ready:** which SD enters the paired SE, and the mean difference with its interval. Questions 3 and 6 are selected response. Notebook notes are optional.

For the Canvas assessment, use `paired_differences` and `paired_result` for Questions 3 (paired SE) and 6 (estimate and interval). These are comparisons within one session, not changes across visits.


## Section 4. Do the separate use-report groups differ?

We now compare **different people**, each contributing one pooled bilateral reward contrast to either **0 days** or **1+ days** for the selected substance in a fixed **365-day** period. Same-wave reporting does not establish intoxication during scanning.

Require MID eligibility **and** a resolved selected-substance report. Someone with an unresolved selected report remains in the first two analyses but cannot enter this comparison. Retain co-users and people with missing other-substance reports. Missing is not zero, and lifetime status cannot replace an unresolved report for this period.

The outcome returns to the **pooled reward contrast**, not the large-minus-small difference. The direction is **any − zero**. **H₀: μany − μzero = 0; Hₐ: μany − μzero ≠ 0.** There is no person-to-person pairing across groups.

### Demo: combine uncertainty from two groups

Welch's method allows unequal population variances. **SE = √(s²any/nany + s²zero/nzero)**. Both groups contribute uncertainty; neither raw group SD alone is the SE. A pooled two-group t test instead assumes a common population variance.

The invented example below shows the difference, Welch SE, and software df. The conservative hand-calculation df is a different approximation, displayed for comparison only. The cell after it switches to your branch's synthetic study data.


In [ ]:
# Invented summaries to show the method; not study data or student-branch answers.
a_n, z_n, a_mean, z_mean, a_sd, z_sd = 20, 30, .14, .10, .20, .10
v_a, v_z = a_sd**2/a_n, z_sd**2/z_n
demo_welch_se = np.sqrt(v_a+v_z)
demo_welch_df = (v_a+v_z)**2 / (v_a**2/(a_n-1) + v_z**2/(z_n-1))
demo_diff = a_mean-z_mean
demo_ci = stats.t.interval(.95,demo_welch_df,loc=demo_diff,scale=demo_welch_se)
print('Invented teaching example (not study data):')
display(pd.DataFrame([dict(difference_PSC=demo_diff,SE=demo_welch_se,Welch_df=demo_welch_df,
    conservative_df=min(a_n-1,z_n-1),low_PSC=demo_ci[0],high_PSC=demo_ci[1])]))


### Your branch: who enters the comparison

This supplied cell applies the selected-substance eligibility rule, adds two rows to `eligibility_audit`, and summarizes each reporting group. From here on, the outputs are your branch's synthetic study results, not the invented example above.


In [ ]:
# Your branch: synthetic study data.
assert df[ELIGIBLE].eq(df[MID].eq(1) & df[RESOLVED].eq(1)).all()
assert df.loc[df[RESOLVED].eq(0),GROUP].isna().all()
days_field = 'su_y_tlfb__mj_ud' if CHOICE == 'cannabis' else 'su_y_tlfb__alc_ud'
resolved = df[RESOLVED].eq(1)
assert df.loc[resolved,GROUP].eq(df.loc[resolved,days_field].gt(0).astype(int)).all()
# Rebuilt from the common rows each time, so re-running this cell is safe.
eligibility_audit = pd.concat([common_eligibility_audit, pd.DataFrame([
    ['MID eligible but chosen report unresolved', int((df[MID].eq(1)&df[RESOLVED].eq(0)).sum())],
    [f'MID + resolved {CHOICE}: independent comparison', int(df[ELIGIBLE].sum())],
], columns=['stage','n'])], ignore_index=True)
display(eligibility_audit)

# Supplied analysis-ready groups: each person contributes one row.
group_data = df.loc[df[ELIGIBLE].eq(1)].copy()
assert group_data.participant_id.is_unique and group_data[[POOLED,GROUP]].notna().all().all()
any_values = group_data.loc[group_data[GROUP].eq(1), POOLED]
zero_values = group_data.loc[group_data[GROUP].eq(0), POOLED]
group_summary = group_data.groupby(GROUP)[POOLED].agg(n='count',mean_PSC='mean',SD='std').rename(index={0:'0 days',1:'1+ days'})
# Show sufficient precision to calculate the signed difference for Canvas.
with pd.option_context('display.precision', 8):
    display(group_summary)
va = any_values.var(ddof=1) / len(any_values)
vz = zero_values.var(ddof=1) / len(zero_values)


### Your Turn 3: calculate the Welch SE and test

The supplied `va` and `vz` are each group's sample variance divided by its n. Calculate `welch_se` as the square root of their sum. Then call `stats.ttest_ind` on `any_values` first and `zero_values` second with **equal_var=False**. Keep the two-sided alternative.

Before running, use the two means in `group_summary` to state the sign that any minus zero should have; your t statistic should share it. The reporting cell checks the SE and test and supplies the corresponding interval.


In [ ]:
# Your Turn 3
welch_se = None  # np.sqrt(...)
welch_test = None  # stats.ttest_ind(..., equal_var=False)


In [ ]:
assert welch_se is not None and welch_test is not None, 'Complete both expressions.'
assert set(group_data.participant_id) == set(df.loc[df[ELIGIBLE].eq(1),'participant_id'])
assert group_data.participant_id.is_unique and group_data[[POOLED,GROUP]].notna().all().all()
assert len(any_values)+len(zero_values) == len(group_data)
na, nz = len(any_values), len(zero_values)
va, vz = any_values.var(ddof=1)/na, zero_values.var(ddof=1)/nz
assert np.isclose(welch_se, np.sqrt(va+vz)), 'Combine the two variance/n contributions.'
welch_df = (va+vz)**2 / (va**2/(na-1)+vz**2/(nz-1))
group_difference = any_values.mean()-zero_values.mean()
welch_ci = stats.t.interval(.95,welch_df,loc=group_difference,scale=welch_se)
assert np.isclose(welch_test.statistic, group_difference/welch_se), ('The Welch test does not match. '
    'Pass any_values first and zero_values second, with equal_var=False.')
assert np.isclose(welch_test.pvalue,2*stats.t.sf(abs(group_difference/welch_se),welch_df), rtol=1e-8, atol=0.0), ('Use equal_var=False '
    'and keep the default two-sided alternative.')
group_comparison = pd.DataFrame([dict(substance=CHOICE,n_any=na,n_zero=nz,estimate_PSC=group_difference,
    SE=welch_se,df=welch_df,low_PSC=welch_ci[0],high_PSC=welch_ci[1],t=welch_test.statistic,p=welch_test.pvalue)])
display(group_comparison)
group_diagnostic = group_data.groupby(GROUP)[POOLED].describe(percentiles=[.01,.5,.99])
display(group_diagnostic)
print('Largest / smallest group SD:', group_summary.SD.max()/group_summary.SD.min())
# Other-substance missingness remains visible; it never removes a main-analysis row.
context_rows = []
for g, records in group_data.groupby(GROUP):
    observed = records[OTHER_GROUP].notna()
    other_n = int(observed.sum())
    positive = int(records.loc[observed,OTHER_GROUP].eq(1).sum())
    context_rows.append(dict(group='1+ days' if g==1 else '0 days',n=len(records),
        other_observed_n=other_n,other_positive_n=positive,other_missing_n=int((~observed).sum()),
        other_positive_percent=100*positive/other_n if other_n else np.nan,
        mean_age=records.ab_g_dyn__visit_age.mean(),mean_cohesion=records.fc_y_fes__cohes_mean.mean()))
context_summary = pd.DataFrame(context_rows)
print('Other substance:',OTHER); display(context_summary)
fig, ax = plt.subplots(figsize=(7,3.5))
ax.hist([zero_values,any_values],bins=30,density=True,label=['0 days','1+ days'],color=['#13678A','#C66A27'],alpha=.7)
ax.set(xlabel='Pooled bilateral contrast (PSC)',ylabel='Density (each group normalized)',title=f'{CHOICE.title()}: inspect spread, tails, and overlap')
ax.legend(); fig.tight_layout(); plt.show()


### Interpret and prepare for Canvas

Report the signed estimate and interval with both group sizes. Does the interval include zero? A nonsignificant test does not establish equality. Welch allows unequal variances; it does not adjust for co-use or make an observational comparison causal. See Vu and Harrington §5.3.4 for confounding.

The supplied context table describes other-substance reports, age, and cohesion. Its co-use percentages use observed other reports as their denominator; they are not covariate adjustments. The group distributions show spread, tails, and overlap. Similar sample SDs do not prove equal population variances.

**Have ready:** the signed difference from the two `group_summary` means. Question 7 is that short calculation, not an evidence statement; Question 8 is selected response. Notebook notes are optional.

For the Canvas assessment, use `eligibility_audit` for Question 2, the two `group_summary` means for the signed-difference calculation in Question 7, and the Welch explanation and `group_comparison` for Question 8.


## Section 5. What do the three comparisons support?

Read each estimate with its interval and research question. A precise task contrast does not imply a substance-group difference. The analyses share observations; they are not three independent replications.

The two figures below describe the same independent-group result. The distributions show differences among people; the difference-with-CI plot emphasizes uncertainty in the group contrast. Identify what each presentation makes easier to see and what it hides.


In [ ]:
design_summary = pd.DataFrame([
    dict(question='One-sample: pooled vs zero',n=len(mid_data),**one_sample_result.iloc[0][['estimate_PSC','low_PSC','high_PSC','t','p']].to_dict()),
    dict(question='Paired: large minus small',n=len(paired_data),**paired_result.iloc[0][['estimate_PSC','low_PSC','high_PSC','t','p']].to_dict()),
    dict(question=f'Independent: {CHOICE} any minus zero',n=len(group_data),**group_comparison.iloc[0][['estimate_PSC','low_PSC','high_PSC','t','p']].to_dict()),
])
display(design_summary)
fig, axes = plt.subplots(1,2,figsize=(9,4))
axes[0].boxplot([zero_values,any_values],positions=[0,1],showfliers=True)
axes[0].scatter([0,1],[zero_values.mean(),any_values.mean()],marker='D',color='#B34B14',label='Mean',zorder=3)
axes[0].set(xticks=[0,1],xticklabels=['0 days','1+ days'],ylabel='Pooled contrast (PSC)',title='People: distributions and means')
axes[0].legend()
axes[1].errorbar(group_difference,0,xerr=[[group_difference-welch_ci[0]],[welch_ci[1]-group_difference]],fmt='D',color='#13678A',capsize=6)
axes[1].axvline(0,color='black',linestyle='--')
axes[1].set(yticks=[],xlabel='Any − zero mean difference (PSC)',title='Group contrast: estimate and 95% CI')
communication_figures = fig
fig.tight_layout(); plt.show()
# Optional private notes only; Canvas is the assessed response.
interpretation_record = pd.DataFrame({'question':design_summary.question,
    'estimate_interval_and_meaning':['','',''], 'assumption_or_limit':['','','']})


### Read each interval against zero and the example difference

A p-value addresses whether the data are compatible with a zero difference. It does not say how large the difference is. Read each 95% interval against two landmarks:

- **zero**: whether the interval includes it matches the two-sided test's decision at .05;
- **−0.02 and +0.02 PSC**: the hypothetical difference worth noticing from Section 1.

At the 95% confidence level under the model, an interval entirely at or beyond +0.02 (or −0.02) supports differences at least that large. An interval entirely between −0.02 and +0.02 supports smaller differences; that is not a formal equivalence test. An interval that crosses a ±0.02 boundary includes both smaller and larger differences. Values outside a 95% interval are not impossible; these conclusions depend on the confidence level and model assumptions.

Before running the next cell, predict where each of your three intervals falls.


In [ ]:
# Supplied: each 95% interval read against zero and the hypothetical ±0.02 PSC example.
def magnitude_relation(lo, hi, example):
    if lo >= example or hi <= -example:
        return f'entirely at/beyond ±{example}'
    if lo > -example and hi < example:
        return f'entirely within ±{example}'
    return f'crosses a ±{example} boundary'

magnitude_check = design_summary[['question','estimate_PSC','low_PSC','high_PSC','p']].copy()
magnitude_check['includes_zero'] = (magnitude_check.low_PSC <= 0) & (magnitude_check.high_PSC >= 0)
magnitude_check['relation_to_example'] = [magnitude_relation(lo, hi, EXAMPLE_DIFFERENCE)
    for lo, hi in zip(magnitude_check.low_PSC, magnitude_check.high_PSC)]
display(magnitude_check)
fig, ax = plt.subplots(figsize=(8,3.6))
ax.axvspan(-EXAMPLE_DIFFERENCE, EXAMPLE_DIFFERENCE, color='#DDEBF1', label=f'Within ±{EXAMPLE_DIFFERENCE} PSC (hypothetical example)')
ax.axvline(0, color='black', linestyle='--', label='Zero difference')
for i, row in enumerate(magnitude_check.itertuples()):
    ax.errorbar(row.estimate_PSC, i, xerr=[[row.estimate_PSC-row.low_PSC],[row.high_PSC-row.estimate_PSC]],
                fmt='o', color='#13678A', capsize=6)
ax.set(yticks=range(len(magnitude_check)), yticklabels=magnitude_check.question, ylim=(-.6, len(magnitude_check)-.4),
       xlabel='Estimate and 95% CI (PSC)', title='Synthetic cohort: each interval against zero and ±0.02 PSC')
ax.invert_yaxis(); ax.legend(loc='lower right', fontsize=8)
magnitude_plot = fig
fig.tight_layout(); plt.show()


### Interpret and prepare for Canvas

The table shows all three intervals. Inspect each one — question, estimate, interval, p-value, and what it says about differences of 0.02 PSC — but write the evidence statement only for the independent comparison. Keep PSC units. A very small p-value shows that the data are hard to reconcile with a zero difference; the estimate and interval answer how large the difference may be. An interval containing zero leaves a range of differences compatible with the model; it does not prove no difference, and it may still include differences as large as the example. The 0.02 PSC value is a hypothetical teaching example, not a justified scientific threshold.

A regional task-fMRI contrast is not a direct measurement of dopamine release, subjective enjoyment, or diagnosis. A small p-value cannot change what the measurement represents. Likewise, the observational group comparison does not establish a causal substance-use effect.

**Have ready:** Canvas Question 9 is the one written evidence statement, for your independent comparison (3–5 sentences), using `group_comparison` and `magnitude_check`. The other two comparisons are for thinking. Notebook notes are optional.

For the Canvas assessment, Question 9 asks you to write this statement for your independent comparison, using `group_comparison` and `magnitude_check`; use this measurement explanation with the results for Question 11 (a supported claim). Optional notes in `interpretation_record` are for your own use, not another submitted assignment.


## Section 6. Finish and retain the results

Complete all **three Your Turns**. Save, restart the kernel, and run the required cells from the top with the same substance choice and settings. The next cell confirms completion and keeps your settings and results in `result_record`; it does not write any files. Keep your completed notebook for reference.

Complete the **Lab 8 Canvas assessment** independently, stating your substance choice in Questions 7 and 9 and using your own outputs there. Record AI use or non-use in Question 12. No notebook upload is required.


In [ ]:
assert len(paired_data) == len(mid_data)
assert CHOICE in ('cannabis','alcohol') and ALPHA == .05
completion_check = pd.DataFrame({'check':['Three coding tasks','Distinct denominators','Two-sided settings','Outputs available','Magnitude check'],
    'passed':[all(v is not None for v in [one_test,paired_test,welch_test]),
              len(group_data)<=len(mid_data),ALPHA==.05,len(design_summary)==3,len(magnitude_check)==3]})
assert completion_check.passed.all()
display(completion_check)
# Settings and results for your records; nothing is written to disk.
result_record = dict(notebook_version=NOTEBOOK_VERSION,dataset_version=DATASET_VERSION,
    csv_sha256_normalized=EXPECTED_SHA256,choice=CHOICE,alpha=ALPHA,alternative='two-sided',example_difference_PSC=EXAMPLE_DIFFERENCE,
    python=platform.python_version(), results=design_summary.to_dict(orient='records'),
    one_sample=one_sample_result.to_dict(orient='records'),paired=paired_result.to_dict(orient='records'),
    independent=group_comparison.to_dict(orient='records'),
    eligibility=eligibility_audit.to_dict(orient='records'),
    magnitude=magnitude_check.to_dict(orient='records'))
print('Notebook', NOTEBOOK_VERSION, '| dataset', DATASET_VERSION, '| choice', CHOICE,
      '| alpha', ALPHA, '| two-sided | results kept in result_record')


## Optional reference

The setup links the local v1.0.1 dictionary and analysis guide. Their field definitions and eligibility rules govern this teaching file. Official-style field names identify **synthetic analogs**; `instructional_` names identify course derivatives. No raw images, repeat scans, or empirical participant observations are present.

The pooled contrast averages the bilateral large and small contrasts equally. Neither hemispheres nor same-session conditions provide a test-retest reliability estimate. Use approved access/privacy and AI rules for real participant data; this synthetic exercise authorizes no transfer of restricted data.

For context, see [ABCD task-fMRI documentation](https://docs.abcdstudy.org/v/6_0_0/documentation/imaging/type_tfmri.html). No live lookup is needed. Further questions about development across visits or a substance-by-reward-size interaction require a new analysis plan; no extra test or response is assigned here.
